# Download Economist METS Metadata

This notebook downloads METS XML files for The Economist Historical Archive. It gets issue IDs from the year-filtered collection pages, downloads `/mets/ECON-YYYY-MMDD.mets.xml` for each issue, validates that the METS file contains physical page files, and saves XML under `data/metadata/`.

Authentication is read from `../auth/nationallizenzen_cookie.json`, relative to this notebook. The file is ignored by Git, so it must be supplied locally and must contain:

```json
{"HHAUTHID": "...", "HANID": "..."}
```

## Configuration

`YEAR_FILTER = None` selects all years from 1843 through 2007. It is advised to use `YEAR_FILTER` = [single_year] and `MAX_ISSUES` = int for bounded tests in the beginning.

In [ ]:
from __future__ import annotations

import json
import re
import time
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse

import requests
from bs4 import BeautifulSoup


BASE_URL = "https://nl-1sub-1uni-2goettingen-1de-1kh4lyi7412e1.zugang.nationallizenzen.de"
START_YEAR = 1843
END_YEAR = 2007
YEAR_FILTER: list[int] | None = None
MAX_ISSUES: int | None = None
DOWNLOAD_METS = True
FORCE_REDOWNLOAD = False

COOKIE_PATH = Path("../auth/nationallizenzen_cookie.json")
OUTPUT_DIR = Path("../../data/metadata").resolve()
REQUEST_TIMEOUT = 60
REQUEST_SLEEP = 0.1

ISSUE_ID_RE = re.compile(r"/id/(ECON-\d{4}-\d{4})(?:[/?#]|$)")
AUTH_FAILURE_MARKERS = (
    "Nationallizenzen Web Anmeldedienst",
    "login.nationallizenzen.de",
    "SAML2/Redirect/SSO",
)

YEARS = list(range(START_YEAR, END_YEAR + 1)) if YEAR_FILTER is None else YEAR_FILTER

assert START_YEAR <= END_YEAR
assert all(START_YEAR <= year <= END_YEAR for year in YEARS)
assert MAX_ISSUES is None or MAX_ISSUES > 0
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Years selected: {YEARS[0]}-{YEARS[-1]} ({len(YEARS)} years)")
print(f"MAX_ISSUES: {MAX_ISSUES}")
print(f"Output directory: {OUTPUT_DIR}")

## Authentication

The cookie file must contain the same two cookie names used by the existing image downloader. The notebook stops with a direct error if either field is missing or the server returns the login page.

In [ ]:
def load_auth_cookies(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cookie file: {path.resolve()}\n"
            'Expected JSON: {"HHAUTHID": "...", "HANID": "..."}'
        )

    cookies = json.loads(path.read_text(encoding="utf-8"))
    required = {"HHAUTHID", "HANID"}
    missing = sorted(required - set(cookies)) if isinstance(cookies, dict) else sorted(required)
    if missing:
        raise ValueError(f"Cookie file is missing required fields {missing}: {path.resolve()}")
    if not all(isinstance(cookies[name], str) and cookies[name].strip() for name in required):
        raise ValueError(f"Cookie fields must be non-empty strings: {path.resolve()}")
    return {name: cookies[name].strip() for name in sorted(required)}


session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 economist-mets-notebook/1.0"})
session.cookies.update(load_auth_cookies(COOKIE_PATH))
print(f"Loaded cookies from {COOKIE_PATH}")

## Helpers

Issue dates come from issue IDs exposed on the collection listing pages. For example, `ECON-1967-1230` is the issue dated `1967-12-30`.

In [ ]:
def collection_url(year: int, page: int = 1) -> str:
    url = f"{BASE_URL}/collection/nlh-eha?{urlencode({'filter[0][year_publish]': year})}"
    return f"{url}&page={page}" if page > 1 else url


def mets_url(issue_id: str) -> str:
    return f"{BASE_URL}/mets/{issue_id}.mets.xml"


def issue_output_path(issue_id: str) -> Path:
    return OUTPUT_DIR / issue_id.split("-")[1] / f"{issue_id}.mets.xml"


def check_response(response: requests.Response, label: str) -> None:
    text_sample = response.text[:5000] if "text" in response.headers.get("Content-Type", "") else ""
    auth_text = response.url + "\n" + text_sample
    if any(marker in auth_text for marker in AUTH_FAILURE_MARKERS):
        raise RuntimeError(
            f"Authentication failed while fetching {label}. Final URL: {response.url}. "
            f"Refresh {COOKIE_PATH} from the logged-in Nationallizenzen session."
        )
    response.raise_for_status()


def get(url: str, label: str) -> requests.Response:
    response = session.get(url, timeout=REQUEST_TIMEOUT, allow_redirects=True)
    check_response(response, label)
    return response


def default_file_count(xml_bytes: bytes) -> int:
    root = ET.fromstring(xml_bytes)
    return sum(
        1
        for file_grp in root.findall(".//{*}fileGrp")
        if file_grp.attrib.get("USE") == "DEFAULT"
        for child in file_grp
        if child.tag.rsplit("}", 1)[-1] == "file"
    )


def validate_mets(issue_id: str, xml_bytes: bytes) -> int:
    if b"<mets:mets" not in xml_bytes[:4096] and b"<mets" not in xml_bytes[:4096]:
        raise ValueError(f"{issue_id}: response is not METS XML")
    count = default_file_count(xml_bytes)
    if count <= 0:
        raise ValueError(f"{issue_id}: METS XML has no DEFAULT files")
    return count


def existing_valid_mets(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        return default_file_count(path.read_bytes()) > 0
    except ET.ParseError:
        return False

## Discover Issues

For each year, the notebook follows the collection pagination and extracts unique `/id/ECON-YYYY-MMDD` links.

In [ ]:
def issue_ids_for_year(year: int) -> list[str]:
    first = get(collection_url(year), f"collection listing for {year}")
    soup = BeautifulSoup(first.text, "html.parser")
    page_numbers = [
        int(match.group(1))
        for link in soup.select("a[href]")
        if (match := re.search(r"[?&]page=(\d+)", link.get("href", "")))
    ]
    last_page = max(page_numbers, default=1)

    issue_ids = set()
    for page in range(1, last_page + 1):
        if page > 1:
            time.sleep(REQUEST_SLEEP)
            soup = BeautifulSoup(get(collection_url(year, page), f"collection listing for {year}, page {page}").text, "html.parser")
        for link in soup.select("a[href]"):
            path = urlparse(urljoin(collection_url(year, page), link["href"])).path
            if match := ISSUE_ID_RE.search(path):
                issue_ids.add(match.group(1))
    return sorted(issue_ids)


issue_ids_by_year = {}
selected_issue_ids = []

for year in YEARS:
    ids = issue_ids_for_year(year)
    issue_ids_by_year[year] = ids
    selected_issue_ids.extend(ids)
    print(f"{year}: {len(ids)} issues")
    if MAX_ISSUES is not None and len(selected_issue_ids) >= MAX_ISSUES:
        break

selected_issue_ids = sorted(dict.fromkeys(selected_issue_ids))
if MAX_ISSUES is not None:
    selected_issue_ids = selected_issue_ids[:MAX_ISSUES]

assert len(selected_issue_ids) == len(set(selected_issue_ids))
print(f"Selected issues: {len(selected_issue_ids)}")

## Download and Log

Existing valid METS files are skipped unless `FORCE_REDOWNLOAD` is enabled.

In [ ]:
download_log = []
download_errors = []

if DOWNLOAD_METS:
    for index, issue_id in enumerate(selected_issue_ids, start=1):
        path = issue_output_path(issue_id)
        try:
            if not FORCE_REDOWNLOAD and existing_valid_mets(path):
                record = {
                    "issue_id": issue_id,
                    "status": "skipped_existing",
                    "path": str(path),
                    "url": mets_url(issue_id),
                    "default_file_count": default_file_count(path.read_bytes()),
                }
            else:
                response = get(mets_url(issue_id), f"METS for {issue_id}")
                count = validate_mets(issue_id, response.content)
                path.parent.mkdir(parents=True, exist_ok=True)
                tmp_path = path.with_suffix(path.suffix + ".tmp")
                tmp_path.write_bytes(response.content)
                tmp_path.replace(path)
                record = {
                    "issue_id": issue_id,
                    "status": "downloaded",
                    "path": str(path),
                    "url": mets_url(issue_id),
                    "final_url": response.url,
                    "default_file_count": count,
                    "bytes": len(response.content),
                }
            download_log.append(record)
            print(f"[{index}/{len(selected_issue_ids)}] {issue_id}: {record['status']} ({record['default_file_count']} pages)")
        except Exception as exc:
            download_errors.append({"issue_id": issue_id, "url": mets_url(issue_id), "error": str(exc)})
            print(f"[{index}/{len(selected_issue_ids)}] {issue_id}: ERROR {exc}")
        time.sleep(REQUEST_SLEEP)
else:
    print("Download skipped")

manifest = {
    "base_url": BASE_URL,
    "year_filter": YEAR_FILTER,
    "years": YEARS,
    "year_issue_counts": {year: len(ids) for year, ids in issue_ids_by_year.items()},
    "issue_count": len(selected_issue_ids),
    "issue_ids": selected_issue_ids,
}

(OUTPUT_DIR / "economist_mets_issue_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
(OUTPUT_DIR / "economist_mets_download_log.json").write_text(json.dumps(download_log, indent=2, sort_keys=True), encoding="utf-8")
(OUTPUT_DIR / "economist_mets_download_errors.json").write_text(json.dumps(download_errors, indent=2, sort_keys=True), encoding="utf-8")

downloaded = sum(row["status"] == "downloaded" for row in download_log)
skipped = sum(row["status"] == "skipped_existing" for row in download_log)
print(f"Summary: {downloaded} downloaded, {skipped} skipped, {len(download_errors)} errors")